In [ ]:
import torch
import torchvision
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)

print("\nGPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working

In [ ]:
!git clone https://github.com/eezkni/MDFS.git

In [ ]:
%cd /kaggle/working/MDFS

In [ ]:
!git rev-parse HEAD
!git status
!git remote -v

In [ ]:
!mkdir -p artifacts data logs reports results figures scripts src tests docs

In [ ]:
!ls -lah

In [ ]:
%pip install -q gdown tqdm pandas scipy matplotlib pytest

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import scipy
import matplotlib
import tqdm
import gdown

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
!nvidia-smi

In [ ]:
import torch

print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess
import platform
import sys
import torch
import torchvision

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True
).strip()

gpu_names = [
    torch.cuda.get_device_name(i)
    for i in range(torch.cuda.device_count())
]

environment = f"""
MDFS REPRODUCTION ENVIRONMENT
=============================

OS:
{platform.platform()}

Python:
{sys.version}

PyTorch:
{torch.__version__}

TorchVision:
{torchvision.__version__}

CUDA available:
{torch.cuda.is_available()}

CUDA runtime:
{torch.version.cuda}

GPUs:
{gpu_names}

Official MDFS commit:
{commit}
"""

Path("reports/environment.txt").write_text(environment)

print(environment)

In [ ]:
!cat train.py

In [ ]:
!grep -n "data/div500\|MDFS_weights\|flist\|torch.save" train.py

In [ ]:
!gdown 1pNTjX5zdwdEzz8yAzTMcx5cvGF_v2k0R -O div500.zip

In [ ]:
!ls -lh div500.zip

In [ ]:
!unzip -l div500.zip | head -40

In [ ]:
!unzip -t div500.zip | tail -5

In [ ]:
!unzip -q div500.zip -d data/

In [ ]:
!ls -ld data/div500

In [ ]:
from pathlib import Path

div500_dir = Path("data/div500")

images = sorted([
    p for p in div500_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
    }
])

print("DIV500 directory:", div500_dir.resolve())
print("Number of images:", len(images))

print("\nFirst 10 images:")
for p in images[:10]:
    print(p.name)

print("\nLast 10 images:")
for p in images[-10:]:
    print(p.name)

assert len(images) == 500, (
    f"Expected exactly 500 images, found {len(images)}"
)

print("\n✅ DIV500 image-count validation PASSED")

In [ ]:
from PIL import Image

bad_images = []

for path in images:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        bad_images.append((path.name, str(e)))

print("Unreadable/corrupt images:", len(bad_images))

if bad_images:
    for item in bad_images[:20]:
        print(item)

assert len(bad_images) == 0

print("✅ All DIV500 images are readable")

In [ ]:
from PIL import Image
import pandas as pd

rows = []

for path in images:
    with Image.open(path) as img:
        rows.append({
            "filename": path.name,
            "width": img.width,
            "height": img.height,
            "mode": img.mode
        })

resolution_df = pd.DataFrame(rows)

display(resolution_df.head(10))

print("\nResolution statistics:")
display(resolution_df[["width", "height"]].describe())

print("\nUnique image modes:")
print(resolution_df["mode"].value_counts())

In [ ]:
import torch
from model import EffNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

effnet = EffNet().to(device)

print("✅ EfficientNet-B7 loaded successfully")
print("GPU:", torch.cuda.get_device_name(0))

trainable_params = sum(
    p.numel()
    for p in effnet.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in effnet.parameters()
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

In [ ]:
del effnet
torch.cuda.empty_cache()

print("Temporary EfficientNet removed from GPU memory.")

In [ ]:
!nvidia-smi

In [ ]:
!git diff -- train.py

In [ ]:
from pathlib import Path

benchmark_code = r'''
import glob
import random
import time
import torch
from PIL import Image

from model import EffNet, APL, prepare_image


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Same models as authors' train.py
vgg = EffNet().to(device)
model = APL().to(device)


# Same training/reference folder
flist = glob.glob("./data/div500/*")

print("Total DIV500 files found:", len(flist))

assert len(flist) == 500, \
    f"Expected 500 reference images, found {len(flist)}"


# Fixed seed ONLY to make our benchmark image subset reproducible.
# It does not alter the real authors' train.py.
random.seed(0)
random.shuffle(flist)

benchmark_files = flist[:10]

print("Benchmark images:", len(benchmark_files))


# Synchronize before starting timer
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.perf_counter()

feats_total = []

with torch.inference_mode():

    for i, f in enumerate(benchmark_files, start=1):

        image = Image.open(f).convert("RGB")

        x = prepare_image(image).to(device)

        feats_x = vgg(x)

        out, w = model(
            feats_x,
            select=1
        )

        feats_total.append(out)

        print(
            f"[{i:02d}/10] "
            f"{f.split('/')[-1]}"
        )


feats_total = torch.cat(
    feats_total,
    dim=0
)


if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start


# Save separately — DO NOT create/overwrite real MDFS_weights.pth
output = (
    "artifacts/"
    "MDFS_weights_10image_benchmark.pth"
)

torch.save(
    feats_total,
    output
)


mean_per_image = elapsed / len(benchmark_files)

estimated_500 = (
    mean_per_image * 500
)


print("\n==============================")
print("10-IMAGE BENCHMARK COMPLETE")
print("==============================")

print(
    f"Elapsed for 10 images: "
    f"{elapsed:.2f} seconds"
)

print(
    f"Mean per image: "
    f"{mean_per_image:.3f} seconds"
)

print(
    f"Estimated 500-image feature time: "
    f"{estimated_500:.2f} seconds"
)

print(
    f"Estimated 500-image feature time: "
    f"{estimated_500 / 60:.2f} minutes"
)

print(
    "Output tensor shape:",
    tuple(feats_total.shape)
)

print(
    "Finite:",
    torch.isfinite(feats_total)
         .all()
         .item()
)

print(
    "Temporary benchmark artifact:",
    output
)
'''

Path(
    "train_benchmark10.py"
).write_text(benchmark_code)

print("Created train_benchmark10.py")

In [ ]:
!sed -n '1,220p' train_benchmark10.py

In [ ]:
!python train_benchmark10.py 2>&1 | tee logs/div500_benchmark10.log

In [ ]:
!rm -f artifacts/MDFS_weights_10image_benchmark.pth

In [ ]:
!time python train.py 2>&1 | tee logs/div500_full_rebuild.log

In [ ]:
!ls -lh MDFS_weights.pth

In [ ]:
!du -h MDFS_weights.pth

In [ ]:
import torch

rebuilt = torch.load(
    "MDFS_weights.pth",
    map_location="cpu",
    weights_only=False
)

print("Type:", type(rebuilt))
print("Shape:", rebuilt.shape)
print("Dtype:", rebuilt.dtype)

print(
    "Finite:",
    torch.isfinite(rebuilt).all().item()
)

print(
    "Contains NaN:",
    torch.isnan(rebuilt).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(rebuilt).any().item()
)

print("Mean:", rebuilt.mean().item())
print("Std:", rebuilt.std().item())

In [ ]:
!cp MDFS_weights.pth artifacts/MDFS_weights_rebuilt.pth

In [ ]:
!sha256sum MDFS_weights.pth
!sha256sum artifacts/MDFS_weights_rebuilt.pth

In [ ]:
!sha256sum artifacts/MDFS_weights_rebuilt.pth \
    > artifacts/MDFS_weights_rebuilt.sha256

In [ ]:
from pathlib import Path
import torch

weight_path = Path(
    "artifacts/MDFS_weights_rebuilt.pth"
)

rebuilt = torch.load(
    weight_path,
    map_location="cpu",
    weights_only=False
)

report = f"""
MDFS REBUILT REFERENCE ARTIFACT
===============================

Source:
Author-provided div500 archive

Reference image count:
500

Output:
{weight_path}

File size:
{weight_path.stat().st_size} bytes

Tensor shape:
{tuple(rebuilt.shape)}

Tensor dtype:
{rebuilt.dtype}

All finite:
{torch.isfinite(rebuilt).all().item()}

Mean:
{rebuilt.mean().item()}

Std:
{rebuilt.std().item()}
"""

Path(
    "reports/reference_reconstruction.txt"
).write_text(report)

print(report)

In [ ]:
%cd /kaggle/working/MDFS

In [ ]:
!pwd

!echo "=== REBUILT WEIGHTS ==="
!ls -lh artifacts/MDFS_weights_rebuilt.pth

!echo
!echo "=== HASH ==="
!sha256sum artifacts/MDFS_weights_rebuilt.pth

!echo
!echo "=== DIV500 COUNT ==="
!find data/div500 -maxdepth 1 -type f | wc -l

In [ ]:
!cp artifacts/MDFS_weights_rebuilt.pth MDFS_weights.pth

In [ ]:
!sha256sum MDFS_weights.pth
!sha256sum artifacts/MDFS_weights_rebuilt.pth

In [ ]:
!python test.py 2>&1 | tee logs/rebuilt_sample_test.log

In [ ]:
!pwd
!ls -lah imgs

In [ ]:
!git ls-files imgs

In [ ]:
!cp test.py test_rebuilt_smoke.py

In [ ]:
!sed -i "s#./imgs/bikes.bmp#./data/div500/0002x3.png#" test_rebuilt_smoke.py

In [ ]:
!grep -n "img_path" test_rebuilt_smoke.py

In [ ]:
!python test_rebuilt_smoke.py 2>&1 | tee logs/rebuilt_sample_test.log

In [ ]:
from pathlib import Path

Path("src/__init__.py").touch()

scorer_code = r'''
import torch
from PIL import Image

from model import EffNet, APL, prepare_image


def cov(m, rowvar=False):
    """
    Exact covariance implementation used by upstream test.py.
    """
    if m.dim() > 2:
        raise ValueError("m has more than 2 dimensions")

    if m.dim() < 2:
        m = m.view(1, -1)

    if not rowvar and m.size(0) != 1:
        m = m.t()

    fact = 1.0 / (m.size(1) - 1)

    m = m - torch.mean(
        m,
        dim=1,
        keepdim=True
    )

    mt = m.t()

    return fact * m.matmul(mt).squeeze()


class MDFSScorer:

    def __init__(
        self,
        weights_path,
        device="cuda"
    ):

        if device == "cuda" and not torch.cuda.is_available():
            device = "cpu"

        self.device = torch.device(device)

        print("MDFS device:", self.device)

        if self.device.type == "cuda":
            print(
                "GPU:",
                torch.cuda.get_device_name(0)
            )

        print("Loading EfficientNet-B7...")

        self.vgg = EffNet().to(
            self.device
        ).eval()

        self.apl = APL().to(
            self.device
        ).eval()

        print("Loading reference features...")

        reference = torch.load(
            weights_path,
            map_location=self.device,
            weights_only=False
        )

        print(
            "Reference tensor:",
            tuple(reference.shape)
        )

        print(
            "Computing reference mean..."
        )

        self.reference_mean = reference.mean(
            0,
            keepdim=True
        )

        print(
            "Computing reference covariance..."
        )

        self.reference_cov = cov(reference)

        # Raw ~613MB reference tensor is no longer needed.
        del reference

        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        print("MDFS scorer ready.")


    @torch.inference_mode()
    def score_pil(self, image):

        image = image.convert("RGB")

        x = prepare_image(
            image
        ).to(self.device)

        features = self.vgg(x)

        out, ps = self.apl(
            features,
            select=0
        )

        # Exact upstream mvg() calculation
        x_cov = cov(out)

        delta = (
            out -
            self.reference_mean
        )

        solution = torch.linalg.solve(
            (
                x_cov +
                self.reference_cov
            ) / 2,
            delta.t()
        )

        distances = delta.mm(
            solution
        )

        distances = torch.diagonal(
            distances
        ).abs().sqrt()

        score_map = distances.reshape(
            ps.shape
        )

        score = (
            score_map * ps
        ).sum() / ps.sum()

        return float(score.item())


    def score_path(self, path):

        with Image.open(path) as image:
            return self.score_pil(image)
'''

Path(
    "src/mdfs_scorer.py"
).write_text(scorer_code)

print("Created src/mdfs_scorer.py")

In [ ]:
from src.mdfs_scorer import MDFSScorer

scorer = MDFSScorer(
    weights_path="artifacts/MDFS_weights_rebuilt.pth",
    device="cuda"
)

score = scorer.score_path(
    "data/div500/0002x3.png"
)

print("\nReusable scorer result:")
print(score)

expected = 18.157730102539062

difference = abs(
    score - expected
)

print("\nExpected from upstream path:")
print(expected)

print("\nAbsolute difference:")
print(difference)

assert difference <= 1e-5, (
    f"Baseline mismatch: {difference}"
)

print("\n✅ BASELINE EQUIVALENCE PASSED")

In [ ]:
!wget -c \
https://www.ponomarenko.info/tid2013/tid2013.rar \
-O /kaggle/working/tid2013.rar

In [ ]:
!ls -lh /kaggle/working/tid2013.rar

In [ ]:
!which 7z

In [ ]:
!7z t /kaggle/working/tid2013.rar | tail -15

In [ ]:
!apt-get update -qq
!apt-get install -y -qq unar

In [ ]:
!which unar

In [ ]:
!which lsar

In [ ]:
!lsar /kaggle/working/tid2013.rar | head -30

In [ ]:
!rm -rf data/TID2013
!mkdir -p data/TID2013

In [ ]:
!unar \
  -f \
  -D \
  -o data/TID2013 \
  /kaggle/working/tid2013.rar \
  2>&1 | tee logs/tid2013_extract_unar.log

In [ ]:
!find data/TID2013 -maxdepth 2 -type d | sort

In [ ]:
from pathlib import Path

TID_ROOT = Path("data/TID2013")

distorted_dir = TID_ROOT / "distorted_images"
reference_dir = TID_ROOT / "reference_images"

distorted_images = sorted([
    p for p in distorted_dir.iterdir()
    if p.is_file() and p.suffix.lower() == ".bmp"
])

reference_images = sorted([
    p for p in reference_dir.iterdir()
    if p.is_file() and p.suffix.lower() == ".bmp"
])

print("Distorted images:", len(distorted_images))
print("Reference images:", len(reference_images))

assert len(distorted_images) == 3000, \
    f"Expected 3000 distorted images, found {len(distorted_images)}"

assert len(reference_images) == 25, \
    f"Expected 25 reference images, found {len(reference_images)}"

print("\n✅ TID2013 IMAGE COUNTS PASSED")

In [ ]:
from PIL import Image

bad_images = []

for path in distorted_images + reference_images:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        bad_images.append((path.name, str(e)))

print("Unreadable images:", len(bad_images))

if bad_images:
    for item in bad_images[:20]:
        print(item)

assert len(bad_images) == 0

print("✅ ALL 3025 TID2013 IMAGES ARE READABLE")

In [ ]:
!find data/TID2013 -maxdepth 3 -type f \
| grep -Ei 'mos|dmos|metric|score|\.txt$' \
| sort

In [ ]:
!head -10 data/TID2013/mos_with_names.txt

In [ ]:
!wc -l data/TID2013/mos_with_names.txt

In [ ]:
import re
import pandas as pd
from pathlib import Path

TID_ROOT = Path("data/TID2013")
distorted_dir = TID_ROOT / "distorted_images"

# Case-insensitive lookup because some archive filenames use uppercase I / BMP
actual_files = {
    p.name.lower(): p
    for p in distorted_dir.iterdir()
    if p.is_file() and p.suffix.lower() == ".bmp"
}

pattern = re.compile(
    r"i(\d{2})_(\d{2})_(\d)\.bmp",
    re.IGNORECASE
)

rows = []

with open(
    TID_ROOT / "mos_with_names.txt",
    "r",
    encoding="utf-8"
) as f:

    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        mos_str, filename = line.split()

        match = pattern.fullmatch(filename)

        if match is None:
            raise ValueError(
                f"Invalid filename at line {line_no}: {filename}"
            )

        key = filename.lower()

        if key not in actual_files:
            raise FileNotFoundError(
                f"Image listed in MOS file not found: {filename}"
            )

        rows.append({
            "filename": filename,
            "image_path": str(actual_files[key].resolve()),
            "mos": float(mos_str),
            "reference_id": int(match.group(1)),
            "distortion_id": int(match.group(2)),
            "distortion_level": int(match.group(3)),
        })

manifest = pd.DataFrame(rows)

print("Manifest rows:", len(manifest))
print("Missing MOS:", manifest["mos"].isna().sum())
print("Duplicate filenames:", manifest["filename"].duplicated().sum())
print("References:", manifest["reference_id"].nunique())
print("Distortions:", manifest["distortion_id"].nunique())
print("Levels:", manifest["distortion_level"].nunique())

print("\nRows per distortion:")
print(manifest.groupby("distortion_id").size())

assert len(manifest) == 3000
assert manifest["mos"].isna().sum() == 0
assert manifest["filename"].duplicated().sum() == 0
assert manifest["reference_id"].nunique() == 25
assert manifest["distortion_id"].nunique() == 24
assert manifest["distortion_level"].nunique() == 5
assert (manifest.groupby("distortion_id").size() == 125).all()

print("\n✅ TID2013 MANIFEST VALIDATION PASSED")

In [ ]:
manifest.to_csv(
    "results/tid2013_manifest.csv",
    index=False
)

print("Saved:")
print("results/tid2013_manifest.csv")

display(manifest.head(10))

In [ ]:
import time
import math
import pandas as pd
import torch

from src.mdfs_scorer import MDFSScorer

# Reuse scorer if it already exists; otherwise recreate it.
if "scorer" not in globals():
    scorer = MDFSScorer(
        weights_path="artifacts/MDFS_weights_rebuilt.pth",
        device="cuda"
    )

# Pick 10 images spread across the whole dataset,
# rather than just the first 10.
indices = [
    round(i * (len(manifest) - 1) / 9)
    for i in range(10)
]

sample10 = manifest.iloc[indices].copy()

results = []

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_total = time.perf_counter()

for n, (_, row) in enumerate(sample10.iterrows(), start=1):

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    score = scorer.score_path(
        row["image_path"]
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    assert math.isfinite(score)

    results.append({
        "filename": row["filename"],
        "mos": row["mos"],
        "reference_id": row["reference_id"],
        "distortion_id": row["distortion_id"],
        "distortion_level": row["distortion_level"],
        "mdfs_raw_score": score,
        "runtime_seconds": elapsed,
    })

    print(
        f"[{n:02d}/10] "
        f"{row['filename']} | "
        f"MOS={row['mos']:.5f} | "
        f"MDFS={score:.6f} | "
        f"{elapsed:.3f}s"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

total_elapsed = time.perf_counter() - start_total

results10 = pd.DataFrame(results)

results10.to_csv(
    "results/tid2013_10_scores.csv",
    index=False
)

mean_time = results10["runtime_seconds"].mean()
estimated_3000 = mean_time * 3000

print("\n==============================")
print("10-IMAGE TEST COMPLETE")
print("==============================")
print(f"Total time: {total_elapsed:.2f} sec")
print(f"Mean/image: {mean_time:.3f} sec")
print(f"Estimated 3000 images: {estimated_3000/60:.2f} min")
print(
    "All scores finite:",
    results10["mdfs_raw_score"]
        .map(math.isfinite)
        .all()
)
print(
    "Saved:",
    "results/tid2013_10_scores.csv"
)

In [ ]:
display(results10)

In [ ]:
import time
import math
import pandas as pd
import torch

# 50 images distributed across the entire manifest
indices50 = [
    round(i * (len(manifest) - 1) / 49)
    for i in range(50)
]

sample50 = manifest.iloc[indices50].copy()

results = []

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_total = time.perf_counter()

for n, (_, row) in enumerate(sample50.iterrows(), start=1):

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    score = scorer.score_path(
        row["image_path"]
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    assert math.isfinite(score)

    results.append({
        "filename": row["filename"],
        "mos": row["mos"],
        "reference_id": row["reference_id"],
        "distortion_id": row["distortion_id"],
        "distortion_level": row["distortion_level"],
        "mdfs_raw_score": score,
        "runtime_seconds": elapsed,
    })

    print(
        f"[{n:02d}/50] "
        f"{row['filename']} | "
        f"MDFS={score:.6f} | "
        f"{elapsed:.3f}s"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

total_elapsed = time.perf_counter() - start_total

results50 = pd.DataFrame(results)

results50.to_csv(
    "results/tid2013_50_scores.csv",
    index=False
)

mean_time = results50["runtime_seconds"].mean()
estimated_full = mean_time * 3000

print("\n==============================")
print("50-IMAGE BENCHMARK COMPLETE")
print("==============================")

print(f"Total time: {total_elapsed:.2f} sec")
print(f"Mean/image: {mean_time:.4f} sec")
print(f"Estimated 3000 images: {estimated_full/60:.2f} min")
print(
    "All scores finite:",
    results50["mdfs_raw_score"]
        .map(math.isfinite)
        .all()
)

In [ ]:
from pathlib import Path

full_eval_code = r'''
import csv
import math
import time
from pathlib import Path

import pandas as pd
import torch

from src.mdfs_scorer import MDFSScorer

MANIFEST_PATH = "results/tid2013_manifest.csv"
OUTPUT_PATH = "results/tid2013_full_scores.csv"
WEIGHTS_PATH = "artifacts/MDFS_weights_rebuilt.pth"

manifest = pd.read_csv(MANIFEST_PATH)

scorer = MDFSScorer(
    weights_path=WEIGHTS_PATH,
    device="cuda"
)

output_path = Path(OUTPUT_PATH)

completed = set()

if output_path.exists():
    existing = pd.read_csv(output_path)
    if "filename" in existing.columns:
        completed = set(existing["filename"].astype(str))

print("Already completed:", len(completed))
print("Total to evaluate:", len(manifest))

fieldnames = [
    "filename",
    "image_path",
    "mos",
    "reference_id",
    "distortion_id",
    "distortion_level",
    "mdfs_raw_score",
    "runtime_seconds"
]

write_header = not output_path.exists() or output_path.stat().st_size == 0

start_all = time.perf_counter()

with output_path.open("a", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    if write_header:
        writer.writeheader()

    for idx, row in manifest.iterrows():

        filename = str(row["filename"])

        if filename in completed:
            continue

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        score = scorer.score_path(
            row["image_path"]
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        if not math.isfinite(score):
            raise RuntimeError(
                f"Non-finite score for {filename}: {score}"
            )

        writer.writerow({
            "filename": filename,
            "image_path": row["image_path"],
            "mos": row["mos"],
            "reference_id": row["reference_id"],
            "distortion_id": row["distortion_id"],
            "distortion_level": row["distortion_level"],
            "mdfs_raw_score": score,
            "runtime_seconds": elapsed
        })

        f.flush()

        done = idx + 1

        if done % 100 == 0 or done == len(manifest):
            print(
                f"[{done}/{len(manifest)}] "
                f"{filename} | "
                f"MDFS={score:.6f} | "
                f"{elapsed:.4f}s"
            )

total_elapsed = time.perf_counter() - start_all

final_df = pd.read_csv(output_path)

print("\n==============================")
print("FULL TID2013 EVALUATION COMPLETE")
print("==============================")
print("Rows:", len(final_df))
print("Unique filenames:", final_df["filename"].nunique())
print(
    "All scores finite:",
    final_df["mdfs_raw_score"]
        .map(math.isfinite)
        .all()
)
print(
    "Mean runtime/image:",
    final_df["runtime_seconds"].mean()
)
print(
    "Total wall time:",
    total_elapsed
)
'''

Path(
    "scripts/evaluate_tid2013_full.py"
).write_text(full_eval_code)

print("Created scripts/evaluate_tid2013_full.py")

In [ ]:
!python scripts/evaluate_tid2013_full.py \
2>&1 | tee logs/tid2013_full_eval.log

In [ ]:
!PYTHONPATH=. python scripts/evaluate_tid2013_full.py \
2>&1 | tee logs/tid2013_full_eval.log

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, kendalltau, pearsonr
from scipy.optimize import curve_fit

df = pd.read_csv(
    "results/tid2013_full_scores.csv"
)

print("Rows:", len(df))

mos = df["mos"].to_numpy(dtype=np.float64)
raw = df["mdfs_raw_score"].to_numpy(dtype=np.float64)

assert len(df) == 3000
assert np.isfinite(mos).all()
assert np.isfinite(raw).all()

# --------------------------------------------------
# MDFS is a distortion/distance score:
# larger raw MDFS = worse quality.
#
# TID2013 MOS:
# larger MOS = better quality.
#
# Therefore use negative MDFS as quality-oriented score.
# --------------------------------------------------

quality_score = -raw


# Rank correlations
raw_srocc = spearmanr(raw, mos).statistic
raw_krocc = kendalltau(raw, mos).statistic

srocc = spearmanr(
    quality_score,
    mos
).statistic

krocc = kendalltau(
    quality_score,
    mos
).statistic


# --------------------------------------------------
# Five-parameter nonlinear mapping used before
# PLCC/RMSE comparison
# --------------------------------------------------

def logistic5(x, b1, b2, b3, b4, b5):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5 -
            1.0 / (1.0 + np.exp(z))
        )
        +
        b4 * x
        +
        b5
    )


p0 = [
    np.max(mos) - np.min(mos),
    1.0,
    np.median(quality_score),
    0.0,
    np.mean(mos)
]

params, _ = curve_fit(
    logistic5,
    quality_score,
    mos,
    p0=p0,
    maxfev=200000
)

mapped = logistic5(
    quality_score,
    *params
)

plcc = pearsonr(
    mapped,
    mos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - mos) ** 2
    )
)


print("\n==============================")
print("TID2013 REPRODUCTION METRICS")
print("==============================")

print(f"Raw SROCC:              {raw_srocc:.6f}")
print(f"Raw KROCC:              {raw_krocc:.6f}")

print()
print(f"Quality-oriented SROCC: {srocc:.6f}")
print(f"Quality-oriented KROCC: {krocc:.6f}")
print(f"PLCC after mapping:     {plcc:.6f}")
print(f"RMSE after mapping:     {rmse:.6f}")

print("\nPaper targets:")
print("SROCC = 0.5363")
print("KROCC = 0.3824")
print("PLCC  = 0.6242")
print("RMSE  = 0.9685")

print("\nDifferences (ours - paper):")
print(f"SROCC: {srocc - 0.5363:+.6f}")
print(f"KROCC: {krocc - 0.3824:+.6f}")
print(f"PLCC:  {plcc  - 0.6242:+.6f}")
print(f"RMSE:  {rmse  - 0.9685:+.6f}")

In [ ]:
from pathlib import Path

report = """
MDFS TID2013 BASELINE REPRODUCTION
=================================

Reference:
Rebuilt from author-provided DIV500 images

Images evaluated:
3000

Paper vs Reproduction
---------------------

SROCC
Paper: 0.536300
Ours:  0.536271
Diff: -0.000029

KROCC
Paper: 0.382400
Ours:  0.382358
Diff: -0.000042

PLCC
Paper: 0.624200
Ours:  0.626722
Diff: +0.002522

RMSE
Paper: 0.968500
Ours:  0.966007
Diff: -0.002493

Raw correlations:
SROCC: -0.536271
KROCC: -0.382358

Runtime
-------
Mean/image: 0.0329691 s
Total wall time: 99.56 s
GPU: 1 x Tesla T4

Validation
----------
Rows: 3000
Unique images: 3000
All scores finite: True
"""

Path(
    "reports/tid2013_baseline_reproduction.txt"
).write_text(report)

print(report)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, kendalltau

df = pd.read_csv(
    "results/tid2013_full_scores.csv"
)

DISTORTION_NAMES = {
    1:  "Additive Gaussian noise",
    2:  "Additive noise in color components",
    3:  "Spatially correlated noise",
    4:  "Masked noise",
    5:  "High frequency noise",
    6:  "Impulse noise",
    7:  "Quantization noise",
    8:  "Gaussian blur",
    9:  "Image denoising",
    10: "JPEG compression",
    11: "JPEG2000 compression",
    12: "JPEG transmission errors",
    13: "JPEG2000 transmission errors",
    14: "Non-eccentricity pattern noise (NEPN)",
    15: "Local block-wise distortions (LBD)",
    16: "Mean shift",
    17: "Contrast change",
    18: "Change of color saturation",
    19: "Multiplicative Gaussian noise",
    20: "Comfort noise",
    21: "Lossy compression of noisy images",
    22: "Image color quantization with dither",
    23: "Chromatic aberrations",
    24: "Sparse sampling and reconstruction",
}

rows = []

for distortion_id in range(1, 25):

    subset = df[
        df["distortion_id"] == distortion_id
    ].copy()

    assert len(subset) == 125

    # MDFS raw score is a distance:
    # larger = worse quality.
    quality_score = -subset[
        "mdfs_raw_score"
    ].to_numpy()

    mos = subset[
        "mos"
    ].to_numpy()

    srocc = spearmanr(
        quality_score,
        mos
    ).statistic

    krocc = kendalltau(
        quality_score,
        mos
    ).statistic

    rows.append({
        "distortion_id": distortion_id,
        "distortion_name":
            DISTORTION_NAMES[distortion_id],
        "images": len(subset),
        "srocc": srocc,
        "krocc": krocc,
    })

distortion_results = pd.DataFrame(rows)

distortion_results.to_csv(
    "results/tid2013_distortion_metrics.csv",
    index=False
)

display(
    distortion_results.round(6)
)

print(
    "\nSaved: "
    "results/tid2013_distortion_metrics.csv"
)

print("\nImportant failure cases:")

display(
    distortion_results[
        distortion_results[
            "distortion_id"
        ].isin([14, 15])
    ].round(6)
)

In [ ]:
import pandas as pd
import numpy as np

paper_srocc = {
    1: 0.8499,
    2: 0.7380,
    3: 0.8150,
    4: 0.6490,
    5: 0.8875,
    6: 0.7703,
    7: 0.8729,
    8: 0.8614,
    9: 0.8752,
    10: 0.8952,
    11: 0.9326,
    12: 0.4233,
    13: 0.5262,
    14: 0.0165,
    15: 0.0889,
    16: 0.1812,
    17: 0.2840,
    18: 0.5557,
    19: 0.7590,
    20: 0.3181,
    21: 0.8458,
    22: 0.8043,
    23: 0.7177,
    24: 0.9196,
}

comparison = distortion_results.copy()

comparison["paper_srocc"] = (
    comparison["distortion_id"]
    .map(paper_srocc)
)

# Keep our signed value
comparison["ours_signed_srocc"] = comparison["srocc"]

# Paper-comparable magnitude
comparison["ours_paper_comparable"] = (
    comparison["srocc"].abs()
)

comparison["difference"] = (
    comparison["ours_paper_comparable"]
    - comparison["paper_srocc"]
)

comparison["abs_difference"] = (
    comparison["difference"].abs()
)

comparison["matches_1e-4"] = (
    comparison["abs_difference"] <= 1e-4
)

comparison.to_csv(
    "results/tid2013_paper_vs_reproduced_distortions.csv",
    index=False
)

display(
    comparison[
        [
            "distortion_id",
            "distortion_name",
            "paper_srocc",
            "ours_signed_srocc",
            "ours_paper_comparable",
            "difference",
            "matches_1e-4",
        ]
    ].round(6)
)

print(
    "\nWithin 1e-4:",
    comparison["matches_1e-4"].sum(),
    "/ 24"
)

In [ ]:
%cd /kaggle/working/MDFS

In [ ]:
!mkdir -p data/CSIQ

!wget -c \
https://s2.smu.edu/~eclarson/csiq/src_imgs.zip \
-O data/CSIQ/src_imgs.zip

!wget -c \
https://s2.smu.edu/~eclarson/csiq/dst_imgs.zip \
-O data/CSIQ/dst_imgs.zip

!wget -c \
https://s2.smu.edu/~eclarson/csiq/csiq.DMOS.xlsx \
-O data/CSIQ/csiq.DMOS.xlsx

In [ ]:
!ls -lh data/CSIQ

In [ ]:
!mkdir -p data/CSIQ/src_imgs
!mkdir -p data/CSIQ/dst_imgs

!unzip -q data/CSIQ/src_imgs.zip \
-d data/CSIQ/src_imgs

!unzip -q data/CSIQ/dst_imgs.zip \
-d data/CSIQ/dst_imgs

In [ ]:
!echo "=== SOURCE TREE ==="
!find data/CSIQ/src_imgs -maxdepth 2 -type f | head -20

!echo
!echo "=== DISTORTED TREE ==="
!find data/CSIQ/dst_imgs -maxdepth 3 -type f | head -30

!echo
!echo "=== COUNTS ==="
!find data/CSIQ/src_imgs -type f | wc -l
!find data/CSIQ/dst_imgs -type f | wc -l

In [ ]:
import pandas as pd

xlsx_path = "data/CSIQ/csiq.DMOS.xlsx"

xls = pd.ExcelFile(xlsx_path)

print("Sheets:")
print(xls.sheet_names)

for sheet in xls.sheet_names:
    print("\n" + "=" * 70)
    print("SHEET:", sheet)
    print("=" * 70)

    temp = pd.read_excel(
        xlsx_path,
        sheet_name=sheet
    )

    print("Shape:", temp.shape)
    print("Columns:")
    print(list(temp.columns))

    display(temp.head(10))

In [ ]:
!find data/CSIQ/dst_imgs \
-mindepth 1 \
-maxdepth 1 \
-type d \
-printf "%f\n" \
| sort

In [ ]:
import pandas as pd

xlsx_path = "data/CSIQ/csiq.DMOS.xlsx"

csiq_raw = pd.read_excel(
    xlsx_path,
    sheet_name="all_by_image",
    header=3
)

print("Shape:", csiq_raw.shape)

print("\nColumns:")
print(list(csiq_raw.columns))

print("\nFirst 10 rows:")
display(csiq_raw.head(10))

print("\nDistortion types:")
print(
    csiq_raw["dst_type"].value_counts(
        dropna=False
    )
)

In [ ]:
valid = csiq_raw[
    csiq_raw["image"].notna()
    & csiq_raw["dst_type"].notna()
    & csiq_raw["dst_lev"].notna()
    & csiq_raw["dmos"].notna()
].copy()

print("Valid scored rows:", len(valid))

print("\nDistortion counts:")
print(valid["dst_type"].value_counts())

print("\nUnique source images:")
print(valid["image"].nunique())

print("\nDMOS range:")
print(
    valid["dmos"].min(),
    "to",
    valid["dmos"].max()
)

display(valid.head(15))

In [ ]:
%%bash

for d in awgn blur contrast fnoise jpeg jpeg2000; do
    echo "=== $d ==="
    find "data/CSIQ/dst_imgs/$d" -maxdepth 1 -type f | head -5
    echo
done

In [ ]:
from pathlib import Path
import pandas as pd

CSIQ_ROOT = Path("data/CSIQ")
DST_ROOT = CSIQ_ROOT / "dst_imgs"

# --------------------------------------------------
# Build lookup of every real distorted image.
# Key = lowercase filename
# Value = actual path with original capitalization
# --------------------------------------------------

actual_files = {}

for p in DST_ROOT.rglob("*.png"):
    actual_files[p.name.lower()] = p

print("Actual distorted PNG files indexed:", len(actual_files))

assert len(actual_files) == 900


# Excel dst_type -> filename token
token_map = {
    "noise": "awgn",
    "jpeg": "jpeg",
    "jpeg 2000": "jpeg2000",
    "fnoise": "fnoise",
    "blur": "blur",
    "contrast": "contrast",
}


rows = []
missing = []

for _, row in valid.iterrows():

    image = str(row["image"]).strip()

    # Excel may interpret names like 1600 as numeric
    if image.endswith(".0"):
        image = image[:-2]

    dst_type = str(row["dst_type"]).strip().lower()
    level = int(row["dst_lev"])

    token = token_map[dst_type]

    # Lowercase lookup makes AWGN/BLUR/JPEG casing irrelevant
    expected_key = (
        f"{image}.{token}.{level}.png"
    ).lower()

    real_path = actual_files.get(expected_key)

    if real_path is None:
        missing.append({
            "image": image,
            "dst_type": dst_type,
            "level": level,
            "expected_key": expected_key
        })
        continue

    rows.append({
        "filename": real_path.name,
        "image_path": str(real_path.resolve()),
        "reference_image": image,
        "dst_type": dst_type,
        "dst_idx": int(row["dst_idx"]),
        "dst_level": level,
        "dmos": float(row["dmos"]),
        "dmos_std": float(row["dmos_std"]),
    })


csiq_manifest = pd.DataFrame(rows)

print("\nManifest rows:", len(csiq_manifest))
print("Missing mappings:", len(missing))
print(
    "Duplicate filenames:",
    csiq_manifest["filename"].duplicated().sum()
)

print("\nPer distortion:")
print(
    csiq_manifest["dst_type"].value_counts()
)

if missing:
    print("\nMissing mappings:")
    display(pd.DataFrame(missing).head(30))


assert len(csiq_manifest) == 866
assert len(missing) == 0
assert csiq_manifest["filename"].duplicated().sum() == 0
assert csiq_manifest["dmos"].notna().all()

print("\n✅ CSIQ MANIFEST VALIDATION PASSED")

csiq_manifest.to_csv(
    "results/csiq_manifest.csv",
    index=False
)

print("Saved: results/csiq_manifest.csv")

display(csiq_manifest.head(15))

In [ ]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Split the 866-image manifest evenly between the two GPUs
# ------------------------------------------------------------

manifest = pd.read_csv("results/csiq_manifest.csv")

gpu0_manifest = manifest.iloc[0::2].reset_index(drop=True)
gpu1_manifest = manifest.iloc[1::2].reset_index(drop=True)

gpu0_manifest.to_csv(
    "results/csiq_gpu0_manifest.csv",
    index=False
)

gpu1_manifest.to_csv(
    "results/csiq_gpu1_manifest.csv",
    index=False
)

print("Total:", len(manifest))
print("GPU 0 images:", len(gpu0_manifest))
print("GPU 1 images:", len(gpu1_manifest))

assert len(gpu0_manifest) == 433
assert len(gpu1_manifest) == 433


# ------------------------------------------------------------
# Create independent worker script
# ------------------------------------------------------------

worker_code = r'''
import argparse
import csv
import math
import os
import time
from pathlib import Path

parser = argparse.ArgumentParser()

parser.add_argument(
    "--gpu",
    required=True
)

parser.add_argument(
    "--input",
    required=True
)

parser.add_argument(
    "--output",
    required=True
)

args = parser.parse_args()


# IMPORTANT:
# Must happen BEFORE importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)


import pandas as pd
import torch

from src.mdfs_scorer import MDFSScorer


print("=" * 60)
print(f"PHYSICAL GPU ASSIGNED: {args.gpu}")
print("CUDA_VISIBLE_DEVICES:",
      os.environ["CUDA_VISIBLE_DEVICES"])
print("Visible CUDA devices:",
      torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1

print(
    "Visible GPU:",
    torch.cuda.get_device_name(0)
)
print("=" * 60)


manifest = pd.read_csv(args.input)

output_path = Path(args.output)

completed = set()

if output_path.exists():

    try:
        existing = pd.read_csv(output_path)

        if "filename" in existing.columns:
            completed = set(
                existing["filename"].astype(str)
            )

    except Exception:
        completed = set()


print(
    f"Worker GPU {args.gpu}: "
    f"{len(manifest)} assigned"
)

print(
    f"Worker GPU {args.gpu}: "
    f"{len(completed)} already completed"
)


scorer = MDFSScorer(
    weights_path=
        "artifacts/MDFS_weights_rebuilt.pth",
    device="cuda"
)


fieldnames = list(manifest.columns) + [
    "mdfs_raw_score",
    "runtime_seconds",
    "worker_gpu"
]


write_header = (
    not output_path.exists()
    or output_path.stat().st_size == 0
)


start_all = time.perf_counter()

processed_now = 0


with output_path.open(
    "a",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    if write_header:
        writer.writeheader()


    for _, row in manifest.iterrows():

        filename = str(
            row["filename"]
        )

        if filename in completed:
            continue


        torch.cuda.synchronize()

        start = time.perf_counter()


        score = scorer.score_path(
            row["image_path"]
        )


        torch.cuda.synchronize()

        elapsed = (
            time.perf_counter()
            - start
        )


        if not math.isfinite(score):
            raise RuntimeError(
                f"Non-finite score: "
                f"{filename} = {score}"
            )


        record = row.to_dict()

        record.update({
            "mdfs_raw_score":
                score,

            "runtime_seconds":
                elapsed,

            "worker_gpu":
                int(args.gpu),
        })


        writer.writerow(record)

        # Persist every completed image
        f.flush()

        processed_now += 1


        if (
            processed_now % 50 == 0
            or processed_now
            == len(manifest)
        ):

            print(
                f"[GPU {args.gpu}] "
                f"{processed_now}/"
                f"{len(manifest)} "
                f"{filename} "
                f"MDFS={score:.6f} "
                f"{elapsed:.4f}s",
                flush=True
            )


total_elapsed = (
    time.perf_counter()
    - start_all
)


result = pd.read_csv(
    output_path
)


print()
print("=" * 60)
print(
    f"GPU {args.gpu} WORKER COMPLETE"
)
print("=" * 60)

print(
    "Rows:",
    len(result)
)

print(
    "Unique filenames:",
    result["filename"].nunique()
)

print(
    "Mean inference time:",
    result["runtime_seconds"].mean()
)

print(
    "Worker wall time:",
    total_elapsed
)

print(
    "All finite:",
    result[
        "mdfs_raw_score"
    ].map(
        math.isfinite
    ).all()
)
'''

Path(
    "scripts/evaluate_csiq_worker.py"
).write_text(worker_code)

print("\n✅ Created dual-GPU worker")
print("scripts/evaluate_csiq_worker.py")

In [ ]:
%%bash

set -e

echo "Starting GPU 0 worker..."
PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 0 \
    --input results/csiq_gpu0_manifest.csv \
    --output results/csiq_gpu0_scores.csv \
    > logs/csiq_gpu0.log 2>&1 &

PID0=$!


echo "Starting GPU 1 worker..."
PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 1 \
    --input results/csiq_gpu1_manifest.csv \
    --output results/csiq_gpu1_scores.csv \
    > logs/csiq_gpu1.log 2>&1 &

PID1=$!


echo
echo "GPU 0 PID: $PID0"
echo "GPU 1 PID: $PID1"


# Give both processes time to load EfficientNet/reference statistics
sleep 4

echo
echo "========== GPU STATUS =========="
nvidia-smi


echo
echo "Waiting for both workers..."

wait $PID0
STATUS0=$?

wait $PID1
STATUS1=$?


echo
echo "GPU 0 exit code: $STATUS0"
echo "GPU 1 exit code: $STATUS1"


echo
echo "========== GPU 0 FINAL LOG =========="
tail -20 logs/csiq_gpu0.log

echo
echo "========== GPU 1 FINAL LOG =========="
tail -20 logs/csiq_gpu1.log

In [ ]:
import pandas as pd
import numpy as np

gpu0 = pd.read_csv(
    "results/csiq_gpu0_scores.csv"
)

gpu1 = pd.read_csv(
    "results/csiq_gpu1_scores.csv"
)

csiq_scores = pd.concat(
    [gpu0, gpu1],
    ignore_index=True
)

print("GPU 0 rows:", len(gpu0))
print("GPU 1 rows:", len(gpu1))
print("Combined rows:", len(csiq_scores))

print(
    "Unique filenames:",
    csiq_scores["filename"].nunique()
)

print(
    "Duplicate filenames:",
    csiq_scores["filename"].duplicated().sum()
)

print(
    "All finite:",
    np.isfinite(
        csiq_scores["mdfs_raw_score"]
    ).all()
)

# Verify against manifest
expected = set(
    csiq_manifest["filename"].astype(str)
)

actual = set(
    csiq_scores["filename"].astype(str)
)

print(
    "Missing from results:",
    len(expected - actual)
)

print(
    "Unexpected results:",
    len(actual - expected)
)

assert len(csiq_scores) == 866
assert csiq_scores["filename"].nunique() == 866
assert csiq_scores["filename"].duplicated().sum() == 0
assert expected == actual
assert np.isfinite(
    csiq_scores["mdfs_raw_score"]
).all()

csiq_scores.to_csv(
    "results/csiq_full_scores.csv",
    index=False
)

print(
    "\n✅ CSIQ DUAL-GPU RESULTS MERGED"
)

In [ ]:
import numpy as np

from scipy.stats import (
    spearmanr,
    kendalltau,
    pearsonr
)

from scipy.optimize import curve_fit


dmos = csiq_scores[
    "dmos"
].to_numpy(dtype=np.float64)

raw = csiq_scores[
    "mdfs_raw_score"
].to_numpy(dtype=np.float64)


# -----------------------------
# Rank correlations
# -----------------------------

srocc = spearmanr(
    raw,
    dmos
).statistic

krocc = kendalltau(
    raw,
    dmos
).statistic


# -----------------------------
# Paper's 5-parameter mapping
# -----------------------------

def logistic5(
    x,
    b1,
    b2,
    b3,
    b4,
    b5
):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5
            -
            1.0 / (
                1.0 + np.exp(z)
            )
        )
        +
        b4 * x
        +
        b5
    )


# Multiple initializations to avoid
# a bad local nonlinear fit.
initial_b2 = [
    1.0,
    -1.0,
    0.1,
    -0.1
]

best_params = None
best_sse = np.inf

for b2 in initial_b2:

    p0 = [
        np.max(dmos) - np.min(dmos),
        b2,
        np.median(raw),
        0.0,
        np.mean(dmos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            raw,
            dmos,
            p0=p0,
            maxfev=300000
        )

        predicted = logistic5(
            raw,
            *params
        )

        sse = np.sum(
            (predicted - dmos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print(
            "Fit attempt failed:",
            b2,
            e
        )


assert best_params is not None

mapped = logistic5(
    raw,
    *best_params
)


plcc = pearsonr(
    mapped,
    dmos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - dmos) ** 2
    )
)


paper = {
    "SROCC": 0.7774,
    "KROCC": 0.5823,
    "PLCC":  0.7907,
    "RMSE":  0.1607,
}


print(
    "\n=============================="
)

print(
    "CSIQ REPRODUCTION METRICS"
)

print(
    "=============================="
)

print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.777400")
print("KROCC: 0.582300")
print("PLCC:  0.790700")
print("RMSE:  0.160700")

print("\nDifferences (ours - paper):")

print(
    f"SROCC: {srocc - paper['SROCC']:+.6f}"
)

print(
    f"KROCC: {krocc - paper['KROCC']:+.6f}"
)

print(
    f"PLCC:  {plcc - paper['PLCC']:+.6f}"
)

print(
    f"RMSE:  {rmse - paper['RMSE']:+.6f}"
)

In [ ]:
from pathlib import Path

report = """
MDFS CSIQ REPRODUCTION
======================

Reference:
Rebuilt from author-provided DIV500 images

Dataset:
CSIQ

Scored images:
866

Hardware:
2 x Tesla T4

Paper vs Reproduction
---------------------

SROCC
Paper: 0.777400
Ours:  0.787133
Diff: +0.009733

KROCC
Paper: 0.582300
Ours:  0.593019
Diff: +0.010719

PLCC
Paper: 0.790700
Ours:  0.832932
Diff: +0.042232

RMSE
Paper: 0.160700
Ours:  0.145277
Diff: -0.015423

Validation:
Rows: 866
Unique images: 866
All scores finite: True
Dual-GPU evaluation: True
"""

Path(
    "reports/csiq_reproduction.txt"
).write_text(report)

print(report)

In [ ]:
!rm -f /kaggle/working/tid2013.rar
!rm -f /kaggle/working/MDFS/div500.zip
!rm -f data/CSIQ/src_imgs.zip
!rm -f data/CSIQ/dst_imgs.zip

!df -h /kaggle/working

In [ ]:
!mkdir -p data/KADID

In [ ]:
!wget -c "https://datasets.vqa.mmsp-kn.de/archives/kadid10k.zip?utm_source=chatgpt.com" \
-O data/KADID/kadid10k.zip

In [ ]:
!ls -lh data/KADID/kadid10k.zip

In [ ]:
!unzip -t data/KADID/kadid10k.zip | tail -10

In [ ]:
!unzip -l data/KADID/kadid10k.zip | head -40

In [ ]:
!mkdir -p data/KADID/extracted

!unzip -q \
data/KADID/kadid10k.zip \
-d data/KADID/extracted

In [ ]:
!find data/KADID/extracted/kadid10k \
-maxdepth 2 \
-type d \
| sort

In [ ]:
from pathlib import Path
import pandas as pd
import re

KADID_ROOT = Path(
    "data/KADID/extracted/kadid10k"
)

image_dir = KADID_ROOT / "images"
dmos_file = KADID_ROOT / "dmos.csv"

pattern = re.compile(
    r"^I\d{2}_\d{2}_\d{2}\.png$",
    re.IGNORECASE
)

real_images = sorted([
    p for p in image_dir.iterdir()
    if p.is_file()
    and pattern.fullmatch(p.name)
])

print("Real KADID distorted images:", len(real_images))

print("\nFirst 10:")
for p in real_images[:10]:
    print(p.name)

print("\nLast 10:")
for p in real_images[-10:]:
    print(p.name)

print("\n=== DMOS CSV ===")

dmos = pd.read_csv(dmos_file)

print("Shape:", dmos.shape)

print("\nColumns:")
print(list(dmos.columns))

display(dmos.head(10))
display(dmos.tail(10))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

KADID_ROOT = Path("data/KADID/extracted/kadid10k")
IMAGE_DIR = KADID_ROOT / "images"

dmos = pd.read_csv(KADID_ROOT / "dmos.csv")

# Index only real images, ignoring ._ macOS metadata files
actual_files = {
    p.name.lower(): p
    for p in IMAGE_DIR.iterdir()
    if p.is_file()
    and re.fullmatch(
        r"I\d{2}_\d{2}_\d{2}\.png",
        p.name,
        flags=re.IGNORECASE
    )
}

print("Indexed real images:", len(actual_files))

rows = []
missing = []

pattern = re.compile(
    r"I(\d{2})_(\d{2})_(\d{2})\.png",
    re.IGNORECASE
)

for _, row in dmos.iterrows():

    filename = str(row["dist_img"]).strip()
    key = filename.lower()

    path = actual_files.get(key)

    if path is None:
        missing.append(filename)
        continue

    m = pattern.fullmatch(filename)

    if m is None:
        raise ValueError(f"Bad filename: {filename}")

    rows.append({
        "filename": filename,
        "image_path": str(path.resolve()),
        "reference_image": str(row["ref_img"]).strip(),
        "reference_id": int(m.group(1)),
        "distortion_id": int(m.group(2)),
        "distortion_level": int(m.group(3)),
        "dmos": float(row["dmos"]),
        "variance": float(row["var"]),
    })

kadid_manifest = pd.DataFrame(rows)

print("\nManifest rows:", len(kadid_manifest))
print("Missing mappings:", len(missing))
print(
    "Duplicate filenames:",
    kadid_manifest["filename"].duplicated().sum()
)

print(
    "Unique references:",
    kadid_manifest["reference_id"].nunique()
)

print(
    "Unique distortion types:",
    kadid_manifest["distortion_id"].nunique()
)

print(
    "Unique levels:",
    kadid_manifest["distortion_level"].nunique()
)

print(
    "DMOS range:",
    kadid_manifest["dmos"].min(),
    "to",
    kadid_manifest["dmos"].max()
)

print("\nImages per distortion:")
print(
    kadid_manifest
    .groupby("distortion_id")
    .size()
)

assert len(actual_files) == 10125
assert len(kadid_manifest) == 10125
assert len(missing) == 0
assert kadid_manifest["filename"].duplicated().sum() == 0
assert kadid_manifest["reference_id"].nunique() == 81
assert kadid_manifest["distortion_id"].nunique() == 25
assert kadid_manifest["distortion_level"].nunique() == 5
assert np.isfinite(kadid_manifest["dmos"]).all()

kadid_manifest.to_csv(
    "results/kadid_manifest.csv",
    index=False
)

print("\n✅ KADID MANIFEST VALIDATION PASSED")
print("Saved: results/kadid_manifest.csv")

In [ ]:
import pandas as pd

manifest = pd.read_csv(
    "results/kadid_manifest.csv"
)

gpu0_manifest = (
    manifest.iloc[0::2]
    .reset_index(drop=True)
)

gpu1_manifest = (
    manifest.iloc[1::2]
    .reset_index(drop=True)
)

gpu0_manifest.to_csv(
    "results/kadid_gpu0_manifest.csv",
    index=False
)

gpu1_manifest.to_csv(
    "results/kadid_gpu1_manifest.csv",
    index=False
)

print("Total:", len(manifest))
print("GPU 0:", len(gpu0_manifest))
print("GPU 1:", len(gpu1_manifest))

assert (
    len(gpu0_manifest)
    + len(gpu1_manifest)
    == 10125
)

In [ ]:
%%bash

set -e

echo "Starting KADID GPU 0..."
PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 0 \
    --input results/kadid_gpu0_manifest.csv \
    --output results/kadid_gpu0_scores.csv \
    > logs/kadid_gpu0.log 2>&1 &

PID0=$!

echo "Starting KADID GPU 1..."
PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 1 \
    --input results/kadid_gpu1_manifest.csv \
    --output results/kadid_gpu1_scores.csv \
    > logs/kadid_gpu1.log 2>&1 &

PID1=$!

echo "GPU0 PID: $PID0"
echo "GPU1 PID: $PID1"

wait $PID0
STATUS0=$?

wait $PID1
STATUS1=$?

echo
echo "GPU 0 exit code: $STATUS0"
echo "GPU 1 exit code: $STATUS1"

echo
echo "===== GPU 0 FINAL ====="
tail -15 logs/kadid_gpu0.log

echo
echo "===== GPU 1 FINAL ====="
tail -15 logs/kadid_gpu1.log

In [ ]:
import pandas as pd
import numpy as np

gpu0 = pd.read_csv(
    "results/kadid_gpu0_scores.csv"
)

gpu1 = pd.read_csv(
    "results/kadid_gpu1_scores.csv"
)

kadid_scores = pd.concat(
    [gpu0, gpu1],
    ignore_index=True
)

print("GPU 0 rows:", len(gpu0))
print("GPU 1 rows:", len(gpu1))
print("Combined rows:", len(kadid_scores))

print(
    "Unique filenames:",
    kadid_scores["filename"].nunique()
)

print(
    "Duplicate filenames:",
    kadid_scores["filename"].duplicated().sum()
)

print(
    "All finite:",
    np.isfinite(
        kadid_scores["mdfs_raw_score"]
    ).all()
)

expected = set(
    kadid_manifest["filename"].astype(str)
)

actual = set(
    kadid_scores["filename"].astype(str)
)

print(
    "Missing results:",
    len(expected - actual)
)

print(
    "Unexpected results:",
    len(actual - expected)
)

assert len(kadid_scores) == 10125
assert kadid_scores["filename"].nunique() == 10125
assert kadid_scores["filename"].duplicated().sum() == 0
assert expected == actual
assert np.isfinite(
    kadid_scores["mdfs_raw_score"]
).all()

kadid_scores.to_csv(
    "results/kadid_full_scores.csv",
    index=False
)

print("\n✅ KADID DUAL-GPU RESULTS MERGED")

In [ ]:
import numpy as np

from scipy.stats import (
    spearmanr,
    kendalltau,
    pearsonr
)

from scipy.optimize import curve_fit


dmos = kadid_scores[
    "dmos"
].to_numpy(dtype=np.float64)

raw = kadid_scores[
    "mdfs_raw_score"
].to_numpy(dtype=np.float64)


# Rank metrics
srocc = spearmanr(
    raw,
    dmos
).statistic

krocc = kendalltau(
    raw,
    dmos
).statistic


# 5-parameter nonlinear regression
def logistic5(x, b1, b2, b3, b4, b5):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5 -
            1.0 / (
                1.0 + np.exp(z)
            )
        )
        +
        b4 * x
        +
        b5
    )


best_params = None
best_sse = np.inf

for b2 in [
    1.0,
    -1.0,
    0.1,
    -0.1
]:

    p0 = [
        np.max(dmos) - np.min(dmos),
        b2,
        np.median(raw),
        0.0,
        np.mean(dmos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            raw,
            dmos,
            p0=p0,
            maxfev=300000
        )

        pred = logistic5(
            raw,
            *params
        )

        sse = np.sum(
            (pred - dmos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print("Fit failed:", b2, e)


assert best_params is not None

mapped = logistic5(
    raw,
    *best_params
)


plcc = pearsonr(
    mapped,
    dmos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - dmos) ** 2
    )
)


paper = {
    "SROCC": 0.5983,
    "KROCC": 0.4238,
    "PLCC":  0.5939,
    "RMSE":  0.8710,
}


print("\n==============================")
print("KADID REPRODUCTION METRICS")
print("==============================")

print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.598300")
print("KROCC: 0.423800")
print("PLCC:  0.593900")
print("RMSE:  0.871000")

print("\nDifferences (ours - paper):")
print(f"SROCC: {srocc - paper['SROCC']:+.6f}")
print(f"KROCC: {krocc - paper['KROCC']:+.6f}")
print(f"PLCC:  {plcc  - paper['PLCC']:+.6f}")
print(f"RMSE:  {rmse  - paper['RMSE']:+.6f}")

In [ ]:
import numpy as np
from scipy.stats import spearmanr, kendalltau

dmos = kadid_scores["dmos"].to_numpy(dtype=np.float64)
raw = kadid_scores["mdfs_raw_score"].to_numpy(dtype=np.float64)

# Preserve signed raw correlations
raw_srocc = spearmanr(raw, dmos).statistic
raw_krocc = kendalltau(raw, dmos).statistic

# KADID: higher DMOS = better quality
# MDFS: higher raw distance = worse quality
quality_score = -raw

srocc = spearmanr(quality_score, dmos).statistic
krocc = kendalltau(quality_score, dmos).statistic

print("Raw signed SROCC:", raw_srocc)
print("Raw signed KROCC:", raw_krocc)

print("\nPaper-comparable:")
print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")

print("\nPaper:")
print("SROCC: 0.598300")
print("KROCC: 0.423800")

print("\nDifferences:")
print(f"SROCC: {srocc - 0.5983:+.6f}")
print(f"KROCC: {krocc - 0.4238:+.6f}")

# PLCC/RMSE from the nonlinear fit you already ran
print("\nPLCC/RMSE:")
print("PLCC:  0.625569")
print("RMSE:  0.844631")

print("\nDifferences:")
print(f"PLCC:  {0.625569 - 0.5939:+.6f}")
print(f"RMSE:  {0.844631 - 0.8710:+.6f}")

In [ ]:
from pathlib import Path

report = """
MDFS KADID-10K REPRODUCTION
===========================

Reference:
Rebuilt from author-provided DIV500 images

Dataset:
KADID-10K

Images evaluated:
10125

Hardware:
2 x Tesla T4

Score orientation:
Raw MDFS higher = worse quality
KADID DMOS higher = better visual quality

Therefore:
Raw SROCC = -0.598352
Raw KROCC = -0.423838

Paper-comparable rank correlations use the opposite orientation.

Paper vs Reproduction
---------------------

SROCC
Paper: 0.598300
Ours:  0.598352
Diff: +0.000052

KROCC
Paper: 0.423800
Ours:  0.423838
Diff: +0.000038

PLCC
Paper: 0.593900
Ours:  0.625569
Diff: +0.031669

RMSE
Paper: 0.871000
Ours:  0.844631
Diff: -0.026369

Validation
----------
Rows: 10125
Unique images: 10125
All scores finite: True
Dual-GPU evaluation: True

Dual-GPU split
--------------
GPU 0: 5063 images
GPU 1: 5062 images

GPU 0 wall time: ~193.56 s
GPU 1 wall time: ~192.37 s
"""

path = Path("reports/kadid_reproduction.txt")
path.write_text(report)

print(report)
print("\nSaved:", path)

In [ ]:
!mkdir -p data/CID2013

!wget -c \
"https://zenodo.org/records/2647033/files/CID2013.7z?download=1" \
-O data/CID2013/CID2013.7z

In [ ]:
!ls -lh data/CID2013/CID2013.7z

In [ ]:
!7z t data/CID2013/CID2013.7z | tail -15

In [ ]:
!7z l data/CID2013/CID2013.7z | head -50

In [ ]:
!mkdir -p data/CID2013/extracted

!7z x -y \
data/CID2013/CID2013.7z \
-odata/CID2013/extracted \
> logs/cid2013_extract.log

In [ ]:
!find data/CID2013/extracted \
-maxdepth 3 \
-type d \
| sort

In [ ]:
from pathlib import Path

CID_ROOT = Path("data/CID2013/extracted")

images = sorted([
    p for p in CID_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
])

print("Image files:", len(images))

print("\nFirst 20 images:")
for p in images[:20]:
    print(p)

print("\nNon-image files:")
for p in CID_ROOT.rglob("*"):
    if (
        p.is_file()
        and p.suffix.lower()
        not in [".jpg", ".jpeg", ".png", ".bmp"]
    ):
        print(p)

In [ ]:
import pandas as pd

xlsx_path = (
    "data/CID2013/extracted/"
    "CID2013 data - version 12112014.xlsx"
)

xls = pd.ExcelFile(xlsx_path)

print("Sheets:")
print(xls.sheet_names)

for sheet in xls.sheet_names:
    print("\n" + "=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)

    df = pd.read_excel(
        xlsx_path,
        sheet_name=sheet,
        header=None
    )

    print("Shape:", df.shape)

    display(
        df.head(20)
    )

In [ ]:
from collections import Counter

group_counts = Counter()

for p in images:
    rel = p.relative_to(CID_ROOT)

    # first directory: IS1, IS2, ...
    group_counts[rel.parts[0]] += 1

print("Images per top-level group:")

for group, count in sorted(group_counts.items()):
    print(group, count)

print("\nTotal:", sum(group_counts.values()))

In [ ]:
import pandas as pd
import re

xlsx_path = (
    "data/CID2013/extracted/"
    "CID2013 data - version 12112014.xlsx"
)

cid_mos = pd.read_excel(
    xlsx_path,
    sheet_name="CID2013 MOS",
    header=0
)

# Normalize ALL whitespace inside column names
cid_mos.columns = [
    re.sub(r"\s+", " ", str(c)).strip()
    for c in cid_mos.columns
]

print("Shape:", cid_mos.shape)

print("\nNormalized columns:")
for c in cid_mos.columns:
    print(repr(c))

print("\nImportant columns:")

display(
    cid_mos[
        [
            "Source_ID",
            "Image set specific MOS",
            "Image set specific MOS with outlier removal",
            "Realigned MOS",
        ]
    ].head(15)
)

print(
    "\nRealigned MOS missing:",
    cid_mos["Realigned MOS"].isna().sum()
)

print(
    "Realigned MOS range:",
    cid_mos["Realigned MOS"].min(),
    "to",
    cid_mos["Realigned MOS"].max()
)

print(
    "Unique Source_ID:",
    cid_mos["Source_ID"].nunique()
)

assert len(cid_mos) == 474
assert cid_mos["Source_ID"].nunique() == 474
assert cid_mos["Realigned MOS"].notna().all()

print("\n✅ CID2013 MOS TABLE VALIDATED")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

CID_ROOT = Path(
    "data/CID2013/extracted"
)

# Index every actual JPG by filename stem.
# Example:
# IS_I_C01_D01.jpg -> is_i_c01_d01
actual_files = {
    p.stem.lower(): p
    for p in CID_ROOT.rglob("*.jpg")
}

print(
    "Actual JPGs indexed:",
    len(actual_files)
)

rows = []
missing = []

for _, row in cid_mos.iterrows():

    source_id = str(
        row["Source_ID"]
    ).strip()

    key = source_id.lower()

    image_path = actual_files.get(key)

    if image_path is None:
        missing.append(source_id)
        continue

    rel = image_path.relative_to(
        CID_ROOT
    )

    rows.append({
        "filename":
            image_path.name,

        "source_id":
            source_id,

        "image_path":
            str(image_path.resolve()),

        "image_set":
            rel.parts[0],

        "cluster":
            rel.parts[1],

        "mos":
            float(
                row["Realigned MOS"]
            )
    })


cid_manifest = pd.DataFrame(
    rows
)

print(
    "\nManifest rows:",
    len(cid_manifest)
)

print(
    "Missing mappings:",
    len(missing)
)

print(
    "Duplicate Source_ID:",
    cid_manifest[
        "source_id"
    ].duplicated().sum()
)

print(
    "Unique files:",
    cid_manifest[
        "filename"
    ].nunique()
)

print(
    "MOS finite:",
    np.isfinite(
        cid_manifest["mos"]
    ).all()
)

print(
    "\nImages per image set:"
)

print(
    cid_manifest[
        "image_set"
    ].value_counts().sort_index()
)

if missing:
    print("\nMissing:")
    print(missing[:30])


assert len(actual_files) == 474
assert len(cid_manifest) == 474
assert len(missing) == 0
assert cid_manifest["source_id"].duplicated().sum() == 0
assert np.isfinite(cid_manifest["mos"]).all()


cid_manifest.to_csv(
    "results/cid2013_manifest.csv",
    index=False
)

print(
    "\n✅ CID2013 MANIFEST VALIDATION PASSED"
)

print(
    "Saved: results/cid2013_manifest.csv"
)

display(
    cid_manifest.head(15)
)

In [ ]:
import pandas as pd

manifest = pd.read_csv(
    "results/cid2013_manifest.csv"
)

gpu0_manifest = (
    manifest.iloc[0::2]
    .reset_index(drop=True)
)

gpu1_manifest = (
    manifest.iloc[1::2]
    .reset_index(drop=True)
)

gpu0_manifest.to_csv(
    "results/cid2013_gpu0_manifest.csv",
    index=False
)

gpu1_manifest.to_csv(
    "results/cid2013_gpu1_manifest.csv",
    index=False
)

print("Total:", len(manifest))
print("GPU 0:", len(gpu0_manifest))
print("GPU 1:", len(gpu1_manifest))

assert len(manifest) == 474
assert len(gpu0_manifest) == 237
assert len(gpu1_manifest) == 237

print("\n✅ CID2013 split ready")

In [ ]:
%%bash

set -e

echo "Starting CID2013 GPU 0..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 0 \
    --input results/cid2013_gpu0_manifest.csv \
    --output results/cid2013_gpu0_scores.csv \
    > logs/cid2013_gpu0.log 2>&1 &

PID0=$!


echo "Starting CID2013 GPU 1..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 1 \
    --input results/cid2013_gpu1_manifest.csv \
    --output results/cid2013_gpu1_scores.csv \
    > logs/cid2013_gpu1.log 2>&1 &

PID1=$!


echo "GPU0 PID: $PID0"
echo "GPU1 PID: $PID1"

wait $PID0
STATUS0=$?

wait $PID1
STATUS1=$?

echo
echo "GPU 0 exit code: $STATUS0"
echo "GPU 1 exit code: $STATUS1"

echo
echo "===== GPU 0 FINAL ====="
tail -15 logs/cid2013_gpu0.log

echo
echo "===== GPU 1 FINAL ====="
tail -15 logs/cid2013_gpu1.log

In [ ]:
import pandas as pd
import numpy as np

gpu0 = pd.read_csv(
    "results/cid2013_gpu0_scores.csv"
)

gpu1 = pd.read_csv(
    "results/cid2013_gpu1_scores.csv"
)

cid_scores = pd.concat(
    [gpu0, gpu1],
    ignore_index=True
)

print("GPU 0 rows:", len(gpu0))
print("GPU 1 rows:", len(gpu1))
print("Combined rows:", len(cid_scores))

print(
    "Unique filenames:",
    cid_scores["filename"].nunique()
)

print(
    "Duplicate filenames:",
    cid_scores["filename"].duplicated().sum()
)

print(
    "All finite:",
    np.isfinite(
        cid_scores["mdfs_raw_score"]
    ).all()
)

expected = set(
    cid_manifest["filename"].astype(str)
)

actual = set(
    cid_scores["filename"].astype(str)
)

print(
    "Missing results:",
    len(expected - actual)
)

print(
    "Unexpected results:",
    len(actual - expected)
)

assert len(cid_scores) == 474
assert cid_scores["filename"].nunique() == 474
assert cid_scores["filename"].duplicated().sum() == 0
assert expected == actual
assert np.isfinite(
    cid_scores["mdfs_raw_score"]
).all()

cid_scores.to_csv(
    "results/cid2013_full_scores.csv",
    index=False
)

print("\n✅ CID2013 DUAL-GPU RESULTS MERGED")

In [ ]:
import numpy as np

from scipy.stats import (
    spearmanr,
    kendalltau,
    pearsonr
)

from scipy.optimize import curve_fit


mos = cid_scores[
    "mos"
].to_numpy(dtype=np.float64)

raw = cid_scores[
    "mdfs_raw_score"
].to_numpy(dtype=np.float64)


# Preserve raw signed correlations
raw_srocc = spearmanr(
    raw,
    mos
).statistic

raw_krocc = kendalltau(
    raw,
    mos
).statistic


# CID2013:
# MOS higher = better quality
# MDFS higher = worse quality
quality_score = -raw

srocc = spearmanr(
    quality_score,
    mos
).statistic

krocc = kendalltau(
    quality_score,
    mos
).statistic


def logistic5(x, b1, b2, b3, b4, b5):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5 -
            1.0 / (
                1.0 + np.exp(z)
            )
        )
        +
        b4 * x
        +
        b5
    )


best_params = None
best_sse = np.inf

for b2 in [
    1.0,
    -1.0,
    0.1,
    -0.1
]:

    p0 = [
        np.max(mos) - np.min(mos),
        b2,
        np.median(quality_score),
        0.0,
        np.mean(mos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            quality_score,
            mos,
            p0=p0,
            maxfev=300000
        )

        pred = logistic5(
            quality_score,
            *params
        )

        sse = np.sum(
            (pred - mos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print("Fit attempt failed:", b2, e)


assert best_params is not None

mapped = logistic5(
    quality_score,
    *best_params
)

plcc = pearsonr(
    mapped,
    mos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - mos) ** 2
    )
)


paper = {
    "SROCC": 0.8571,
    "KROCC": 0.6706,
    "PLCC":  0.8717,
    "RMSE":  11.0931,
}


print("\n==============================")
print("CID2013 REPRODUCTION METRICS")
print("==============================")

print(f"Raw signed SROCC: {raw_srocc:.6f}")
print(f"Raw signed KROCC: {raw_krocc:.6f}")

print("\nPaper-comparable:")
print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.857100")
print("KROCC: 0.670600")
print("PLCC:  0.871700")
print("RMSE:  11.093100")

print("\nDifferences (ours - paper):")
print(f"SROCC: {srocc - paper['SROCC']:+.6f}")
print(f"KROCC: {krocc - paper['KROCC']:+.6f}")
print(f"PLCC:  {plcc  - paper['PLCC']:+.6f}")
print(f"RMSE:  {rmse  - paper['RMSE']:+.6f}")

In [ ]:
from pathlib import Path

report = """
MDFS CID2013 REPRODUCTION
=========================

Dataset:
CID2013

Images evaluated:
474

Hardware:
2 x Tesla T4

MOS:
Realigned MOS

Paper vs Reproduction
---------------------

SROCC
Paper: 0.857100
Ours:  0.857070
Diff: -0.000030

KROCC
Paper: 0.670600
Ours:  0.670565
Diff: -0.000035

PLCC
Paper: 0.871700
Ours:  0.871736
Diff: +0.000036

RMSE
Paper: 11.093100
Ours:  11.093217
Diff: +0.000117

Validation:
Rows: 474
Unique images: 474
All scores finite: True
Dual-GPU evaluation: True
"""

Path("reports/cid2013_reproduction.txt").write_text(report)

print(report)

In [ ]:
!rm -f data/CID2013/CID2013.7z
!rm -f data/KADID/kadid10k.zip

!df -h /kaggle/working

In [ ]:
!mkdir -p data/MDIVL

!wget -c \
"http://ivc.uwaterloo.ca/database/IQADataset/MDIVL.tar" \
-O data/MDIVL/MDIVL.tar

In [ ]:
!ls -lh data/MDIVL/MDIVL.tar

In [ ]:
!tar -tf data/MDIVL/MDIVL.tar | head -40

In [ ]:
!tar -tf data/MDIVL/MDIVL.tar | wc -l

In [ ]:
!mkdir -p data/MDIVL/extracted

!tar -xf \
data/MDIVL/MDIVL.tar \
-C data/MDIVL/extracted

In [ ]:
from pathlib import Path

MDIVL_ROOT = Path("data/MDIVL/extracted")

jpgs = sorted(MDIVL_ROOT.rglob("*.jpg"))
pngs = sorted(MDIVL_ROOT.rglob("*.png"))

print("JPG files:", len(jpgs))
print("PNG files:", len(pngs))
print("Total images:", len(jpgs) + len(pngs))

print("\nFirst JPGs:")
for p in jpgs[:10]:
    print(p)

print("\nReference PNGs:")
for p in pngs[:20]:
    print(p)

In [ ]:
!wget -q \
"https://raw.githubusercontent.com/icbcbicc/IQA-Dataset/bfc87eeb6b6a9862c13da2a2d765ea7c136f6789/iqadataset/csv/MDIVL.txt" \
-O data/MDIVL/MDIVL_scores.csv

In [ ]:
import pandas as pd

md_scores = pd.read_csv(
    "data/MDIVL/MDIVL_scores.csv"
)

print("Shape:", md_scores.shape)
print("\nColumns:")
print(list(md_scores.columns))

print("\nUnique distorted images:")
print(md_scores["dis_img_path"].nunique())

print("\nUnique references:")
print(md_scores["ref_img_path"].nunique())

print("\nScore range:")
print(
    md_scores["score"].min(),
    "to",
    md_scores["score"].max()
)

display(md_scores.head(10))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

EXTRACT_ROOT = Path("data/MDIVL/extracted")

# CSV paths are relative to the extraction root:
# MDIVL/MDIVL/Data_Blur_JPEG/...
rows = []
missing = []

for _, row in md_scores.iterrows():

    relative_path = Path(
        str(row["dis_img_path"]).strip()
    )

    image_path = EXTRACT_ROOT / relative_path

    if not image_path.exists():
        missing.append(str(relative_path))
        continue

    rows.append({
        "filename": image_path.name,
        "image_path": str(image_path.resolve()),
        "distortion_type": str(row["dis_type"]).strip(),
        "reference_path": str(row["ref_img_path"]).strip(),
        "mos": float(row["score"]),
    })


mdivl_manifest = pd.DataFrame(rows)

print("Manifest rows:", len(mdivl_manifest))
print("Missing mappings:", len(missing))

print(
    "Unique filenames:",
    mdivl_manifest["filename"].nunique()
)

print(
    "Duplicate full image paths:",
    mdivl_manifest["image_path"].duplicated().sum()
)

print(
    "Unique references:",
    mdivl_manifest["reference_path"].nunique()
)

print(
    "MOS finite:",
    np.isfinite(
        mdivl_manifest["mos"]
    ).all()
)

print(
    "MOS range:",
    mdivl_manifest["mos"].min(),
    "to",
    mdivl_manifest["mos"].max()
)

if missing:
    print("\nMissing examples:")
    print(missing[:20])


assert len(mdivl_manifest) == 750
assert len(missing) == 0
assert mdivl_manifest["image_path"].duplicated().sum() == 0
assert mdivl_manifest["reference_path"].nunique() == 10
assert np.isfinite(mdivl_manifest["mos"]).all()

mdivl_manifest.to_csv(
    "results/mdivl_manifest.csv",
    index=False
)

print("\n✅ MDIVL MANIFEST VALIDATION PASSED")
print("Saved: results/mdivl_manifest.csv")

display(mdivl_manifest.head(15))

In [ ]:
import pandas as pd

manifest = pd.read_csv(
    "results/mdivl_manifest.csv"
)

gpu0_manifest = (
    manifest.iloc[0::2]
    .reset_index(drop=True)
)

gpu1_manifest = (
    manifest.iloc[1::2]
    .reset_index(drop=True)
)

gpu0_manifest.to_csv(
    "results/mdivl_gpu0_manifest.csv",
    index=False
)

gpu1_manifest.to_csv(
    "results/mdivl_gpu1_manifest.csv",
    index=False
)

print("Total:", len(manifest))
print("GPU 0:", len(gpu0_manifest))
print("GPU 1:", len(gpu1_manifest))

assert len(manifest) == 750
assert len(gpu0_manifest) == 375
assert len(gpu1_manifest) == 375

print("\n✅ MDIVL split ready")

In [ ]:
%%bash

set -e

echo "Starting MDIVL GPU 0..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 0 \
    --input results/mdivl_gpu0_manifest.csv \
    --output results/mdivl_gpu0_scores.csv \
    > logs/mdivl_gpu0.log 2>&1 &

PID0=$!


echo "Starting MDIVL GPU 1..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 1 \
    --input results/mdivl_gpu1_manifest.csv \
    --output results/mdivl_gpu1_scores.csv \
    > logs/mdivl_gpu1.log 2>&1 &

PID1=$!


echo "GPU0 PID: $PID0"
echo "GPU1 PID: $PID1"

wait $PID0
STATUS0=$?

wait $PID1
STATUS1=$?


echo
echo "GPU 0 exit code: $STATUS0"
echo "GPU 1 exit code: $STATUS1"

echo
echo "===== GPU 0 FINAL ====="
tail -15 logs/mdivl_gpu0.log

echo
echo "===== GPU 1 FINAL ====="
tail -15 logs/mdivl_gpu1.log

In [ ]:
import pandas as pd
import numpy as np

gpu0 = pd.read_csv(
    "results/mdivl_gpu0_scores.csv"
)

gpu1 = pd.read_csv(
    "results/mdivl_gpu1_scores.csv"
)

mdivl_scores = pd.concat(
    [gpu0, gpu1],
    ignore_index=True
)

print("GPU 0 rows:", len(gpu0))
print("GPU 1 rows:", len(gpu1))
print("Combined rows:", len(mdivl_scores))

print(
    "Unique filenames:",
    mdivl_scores["filename"].nunique()
)

print(
    "Duplicate filenames:",
    mdivl_scores["filename"].duplicated().sum()
)

print(
    "All finite:",
    np.isfinite(
        mdivl_scores["mdfs_raw_score"]
    ).all()
)

expected = set(
    mdivl_manifest["filename"].astype(str)
)

actual = set(
    mdivl_scores["filename"].astype(str)
)

print(
    "Missing results:",
    len(expected - actual)
)

print(
    "Unexpected results:",
    len(actual - expected)
)

assert len(mdivl_scores) == 750
assert mdivl_scores["filename"].nunique() == 750
assert mdivl_scores["filename"].duplicated().sum() == 0
assert expected == actual
assert np.isfinite(
    mdivl_scores["mdfs_raw_score"]
).all()

mdivl_scores.to_csv(
    "results/mdivl_full_scores.csv",
    index=False
)

print("\n✅ MDIVL DUAL-GPU RESULTS MERGED")

In [ ]:
import numpy as np

from scipy.stats import (
    spearmanr,
    kendalltau,
    pearsonr
)

from scipy.optimize import curve_fit


mos = mdivl_scores[
    "mos"
].to_numpy(dtype=np.float64)

raw = mdivl_scores[
    "mdfs_raw_score"
].to_numpy(dtype=np.float64)


# Raw signed correlations
raw_srocc = spearmanr(
    raw,
    mos
).statistic

raw_krocc = kendalltau(
    raw,
    mos
).statistic


# Higher value now means better quality
quality_score = -raw

srocc = spearmanr(
    quality_score,
    mos
).statistic

krocc = kendalltau(
    quality_score,
    mos
).statistic


def logistic5(x, b1, b2, b3, b4, b5):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5 -
            1.0 / (
                1.0 + np.exp(z)
            )
        )
        +
        b4 * x
        +
        b5
    )


best_params = None
best_sse = np.inf

for b2 in [
    1.0,
    -1.0,
    0.1,
    -0.1
]:

    p0 = [
        np.max(mos) - np.min(mos),
        b2,
        np.median(quality_score),
        0.0,
        np.mean(mos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            quality_score,
            mos,
            p0=p0,
            maxfev=300000
        )

        pred = logistic5(
            quality_score,
            *params
        )

        sse = np.sum(
            (pred - mos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print("Fit failed:", b2, e)


assert best_params is not None

mapped = logistic5(
    quality_score,
    *best_params
)

plcc = pearsonr(
    mapped,
    mos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - mos) ** 2
    )
)


paper = {
    "SROCC": 0.7890,
    "KROCC": 0.5911,
    "PLCC":  0.7953,
    "RMSE":  14.4779,
}


print("\n==============================")
print("MDIVL REPRODUCTION METRICS")
print("==============================")

print(f"Raw signed SROCC: {raw_srocc:.6f}")
print(f"Raw signed KROCC: {raw_krocc:.6f}")

print("\nPaper-comparable:")
print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.789000")
print("KROCC: 0.591100")
print("PLCC:  0.795300")
print("RMSE:  14.477900")

print("\nDifferences (ours - paper):")
print(f"SROCC: {srocc - paper['SROCC']:+.6f}")
print(f"KROCC: {krocc - paper['KROCC']:+.6f}")
print(f"PLCC:  {plcc  - paper['PLCC']:+.6f}")
print(f"RMSE:  {rmse  - paper['RMSE']:+.6f}")

In [ ]:
from pathlib import Path

report = """
MDFS MDIVL REPRODUCTION
=======================

Dataset:
MDIVL

Images evaluated:
750

Reference images:
10

Hardware:
2 x Tesla T4

Score orientation:
Raw MDFS higher = worse quality
MDIVL subjective score higher = better quality

Paper vs Reproduction
---------------------

SROCC
Paper: 0.789000
Ours:  0.787468
Diff: -0.001532

KROCC
Paper: 0.591100
Ours:  0.588678
Diff: -0.002422

PLCC
Paper: 0.795300
Ours:  0.795339
Diff: +0.000039

RMSE
Paper: 14.477900
Ours:  14.476242
Diff: -0.001658

Validation:
Rows: 750
Unique images: 750
All scores finite: True
Dual-GPU evaluation: True

Dual-GPU split:
GPU 0: 375
GPU 1: 375
"""

Path(
    "reports/mdivl_reproduction.txt"
).write_text(report)

print(report)

In [ ]:
!mkdir -p data/MDLIVE

!wget -c \
"http://ivc.uwaterloo.ca/database/IQADataset/LIVE_MD.tar" \
-O data/MDLIVE/LIVE_MD.tar

!ls -lh data/MDLIVE/LIVE_MD.tar

In [ ]:
!tar -tf data/MDLIVE/LIVE_MD.tar | head -40
!tar -tf data/MDLIVE/LIVE_MD.tar | wc -l

In [ ]:
!mkdir -p data/MDLIVE/extracted

!tar -xf \
data/MDLIVE/LIVE_MD.tar \
-C data/MDLIVE/extracted

In [ ]:
from pathlib import Path

MDLIVE_ROOT = Path("data/MDLIVE/extracted")

bmps = sorted(MDLIVE_ROOT.rglob("*.bmp"))

print("Total BMP files:", len(bmps))

print("\nFirst 20 BMPs:")
for p in bmps[:20]:
    print(p)

print("\nDirectories containing BMP files:")
dirs = sorted(set(str(p.parent) for p in bmps))
for d in dirs:
    print(d)

In [ ]:
!wget -q \
"https://raw.githubusercontent.com/icbcbicc/IQA-Dataset/bfc87eeb6b6a9862c13da2a2d765ea7c136f6789/iqadataset/csv/LIVE_MD.txt" \
-O data/MDLIVE/LIVE_MD_scores.csv

In [ ]:
import pandas as pd

mdlive_scores = pd.read_csv(
    "data/MDLIVE/LIVE_MD_scores.csv"
)

print("Shape:", mdlive_scores.shape)

print("\nColumns:")
print(list(mdlive_scores.columns))

print(
    "\nUnique distorted images:",
    mdlive_scores["dis_img_path"].nunique()
)

print(
    "Unique references:",
    mdlive_scores["ref_img_path"].nunique()
)

print(
    "Score range:",
    mdlive_scores["score"].min(),
    "to",
    mdlive_scores["score"].max()
)

display(mdlive_scores.head(10))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

MDLIVE_ROOT = Path(
    "data/MDLIVE/extracted"
)

rows = []
missing_distorted = []
missing_references = []

for _, row in mdlive_scores.iterrows():

    distorted_rel = Path(
        str(row["dis_img_path"]).strip()
    )

    reference_rel = Path(
        str(row["ref_img_path"]).strip()
    )

    distorted_path = (
        MDLIVE_ROOT / distorted_rel
    )

    reference_path = (
        MDLIVE_ROOT / reference_rel
    )

    if not distorted_path.exists():
        missing_distorted.append(
            str(distorted_rel)
        )
        continue

    if not reference_path.exists():
        missing_references.append(
            str(reference_rel)
        )

    rows.append({
        "filename":
            distorted_path.name,

        "image_path":
            str(distorted_path.resolve()),

        "distortion_type":
            str(row["dis_type"]).strip(),

        "reference_image":
            reference_path.name,

        "reference_path":
            str(reference_path.resolve()),

        "mos":
            float(row["score"]),
    })


mdlive_manifest = pd.DataFrame(rows)


print(
    "Manifest rows:",
    len(mdlive_manifest)
)

print(
    "Missing distorted mappings:",
    len(missing_distorted)
)

print(
    "Missing reference mappings:",
    len(missing_references)
)

print(
    "Unique full image paths:",
    mdlive_manifest[
        "image_path"
    ].nunique()
)

print(
    "Duplicate full image paths:",
    mdlive_manifest[
        "image_path"
    ].duplicated().sum()
)

print(
    "Unique references:",
    mdlive_manifest[
        "reference_path"
    ].nunique()
)

print(
    "Scores finite:",
    np.isfinite(
        mdlive_manifest["mos"]
    ).all()
)

print(
    "Score range:",
    mdlive_manifest["mos"].min(),
    "to",
    mdlive_manifest["mos"].max()
)


if missing_distorted:
    print("\nMissing distorted examples:")
    print(missing_distorted[:20])

if missing_references:
    print("\nMissing reference examples:")
    print(missing_references[:20])


assert len(mdlive_manifest) == 450
assert len(missing_distorted) == 0
assert len(missing_references) == 0

assert (
    mdlive_manifest[
        "image_path"
    ].duplicated().sum()
    == 0
)

assert (
    mdlive_manifest[
        "reference_path"
    ].nunique()
    == 15
)

assert np.isfinite(
    mdlive_manifest["mos"]
).all()


mdlive_manifest.to_csv(
    "results/mdlive_manifest.csv",
    index=False
)


print(
    "\n✅ MDLIVE MANIFEST VALIDATION PASSED"
)

print(
    "Saved: results/mdlive_manifest.csv"
)

display(
    mdlive_manifest.head(15)
)

In [ ]:
import pandas as pd

manifest = pd.read_csv(
    "results/mdlive_manifest.csv"
)

gpu0_manifest = (
    manifest.iloc[0::2]
    .reset_index(drop=True)
)

gpu1_manifest = (
    manifest.iloc[1::2]
    .reset_index(drop=True)
)


gpu0_manifest.to_csv(
    "results/mdlive_gpu0_manifest.csv",
    index=False
)

gpu1_manifest.to_csv(
    "results/mdlive_gpu1_manifest.csv",
    index=False
)


print("Total:", len(manifest))
print("GPU 0:", len(gpu0_manifest))
print("GPU 1:", len(gpu1_manifest))


assert len(manifest) == 450
assert len(gpu0_manifest) == 225
assert len(gpu1_manifest) == 225

print(
    "\n✅ MDLIVE dual-GPU split ready"
)

In [ ]:
%%bash

set -e

echo "Starting MDLIVE GPU 0..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 0 \
    --input results/mdlive_gpu0_manifest.csv \
    --output results/mdlive_gpu0_scores.csv \
    > logs/mdlive_gpu0.log 2>&1 &

PID0=$!


echo "Starting MDLIVE GPU 1..."

PYTHONPATH=. \
python scripts/evaluate_csiq_worker.py \
    --gpu 1 \
    --input results/mdlive_gpu1_manifest.csv \
    --output results/mdlive_gpu1_scores.csv \
    > logs/mdlive_gpu1.log 2>&1 &

PID1=$!


echo "GPU0 PID: $PID0"
echo "GPU1 PID: $PID1"


wait $PID0
STATUS0=$?

wait $PID1
STATUS1=$?


echo
echo "GPU 0 exit code: $STATUS0"
echo "GPU 1 exit code: $STATUS1"


echo
echo "===== GPU 0 FINAL ====="
tail -15 logs/mdlive_gpu0.log


echo
echo "===== GPU 1 FINAL ====="
tail -15 logs/mdlive_gpu1.log

In [ ]:
import pandas as pd
import numpy as np

gpu0 = pd.read_csv(
    "results/mdlive_gpu0_scores.csv"
)

gpu1 = pd.read_csv(
    "results/mdlive_gpu1_scores.csv"
)

mdlive_full = pd.concat(
    [gpu0, gpu1],
    ignore_index=True
)

print("GPU 0 rows:", len(gpu0))
print("GPU 1 rows:", len(gpu1))
print("Combined rows:", len(mdlive_full))

print(
    "Unique image paths:",
    mdlive_full["image_path"].nunique()
)

print(
    "Duplicate image paths:",
    mdlive_full["image_path"].duplicated().sum()
)

print(
    "All finite:",
    np.isfinite(
        mdlive_full["mdfs_raw_score"]
    ).all()
)

expected = set(
    mdlive_manifest["image_path"].astype(str)
)

actual = set(
    mdlive_full["image_path"].astype(str)
)

print(
    "Missing results:",
    len(expected - actual)
)

print(
    "Unexpected results:",
    len(actual - expected)
)

assert len(mdlive_full) == 450
assert mdlive_full["image_path"].nunique() == 450
assert mdlive_full["image_path"].duplicated().sum() == 0
assert expected == actual
assert np.isfinite(
    mdlive_full["mdfs_raw_score"]
).all()

mdlive_full.to_csv(
    "results/mdlive_full_scores.csv",
    index=False
)

print("\n✅ MDLIVE RESULTS MERGED")

In [ ]:
import numpy as np

from scipy.stats import (
    spearmanr,
    kendalltau,
    pearsonr
)

from scipy.optimize import curve_fit


mos = mdlive_full[
    "mos"
].to_numpy(dtype=np.float64)

raw = mdlive_full[
    "mdfs_raw_score"
].to_numpy(dtype=np.float64)


# Raw signed correlations
raw_srocc = spearmanr(
    raw,
    mos
).statistic

raw_krocc = kendalltau(
    raw,
    mos
).statistic


# MDLIVE subjective score:
# higher = better quality
# MDFS distance:
# higher = worse quality
quality_score = -raw


srocc = spearmanr(
    quality_score,
    mos
).statistic

krocc = kendalltau(
    quality_score,
    mos
).statistic


def logistic5(x, b1, b2, b3, b4, b5):

    z = np.clip(
        b2 * (x - b3),
        -60,
        60
    )

    return (
        b1 * (
            0.5 -
            1.0 / (
                1.0 + np.exp(z)
            )
        )
        +
        b4 * x
        +
        b5
    )


best_params = None
best_sse = np.inf


for b2 in [
    1.0,
    -1.0,
    0.1,
    -0.1
]:

    p0 = [
        np.max(mos) - np.min(mos),
        b2,
        np.median(quality_score),
        0.0,
        np.mean(mos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            quality_score,
            mos,
            p0=p0,
            maxfev=300000
        )

        pred = logistic5(
            quality_score,
            *params
        )

        sse = np.sum(
            (pred - mos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print("Fit failed:", b2, e)


assert best_params is not None


mapped = logistic5(
    quality_score,
    *best_params
)

plcc = pearsonr(
    mapped,
    mos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - mos) ** 2
    )
)


paper = {
    "SROCC": 0.7579,
    "KROCC": 0.5623,
    "PLCC":  0.8226,
    "RMSE":  10.7534,
}


print("\n==============================")
print("MDLIVE REPRODUCTION METRICS")
print("==============================")

print(f"Raw signed SROCC: {raw_srocc:.6f}")
print(f"Raw signed KROCC: {raw_krocc:.6f}")

print("\nPaper-comparable:")
print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.757900")
print("KROCC: 0.562300")
print("PLCC:  0.822600")
print("RMSE:  10.753400")

print("\nDifferences (ours - paper):")
print(f"SROCC: {srocc - paper['SROCC']:+.6f}")
print(f"KROCC: {krocc - paper['KROCC']:+.6f}")
print(f"PLCC:  {plcc  - paper['PLCC']:+.6f}")
print(f"RMSE:  {rmse  - paper['RMSE']:+.6f}")

In [ ]:
# MDLIVE orientation correction
# MDLIVE subjective score behaves as distortion/DMOS:
# higher score = worse quality
# MDFS raw distance:
# higher score = worse quality
#
# Therefore DO NOT negate raw.

quality_score = raw

srocc = spearmanr(
    quality_score,
    mos
).statistic

krocc = kendalltau(
    quality_score,
    mos
).statistic

# Refit PLCC/RMSE using raw orientation
best_params = None
best_sse = np.inf

for b2 in [1.0, -1.0, 0.1, -0.1]:

    p0 = [
        np.max(mos) - np.min(mos),
        b2,
        np.median(quality_score),
        0.0,
        np.mean(mos)
    ]

    try:

        params, _ = curve_fit(
            logistic5,
            quality_score,
            mos,
            p0=p0,
            maxfev=300000
        )

        pred = logistic5(
            quality_score,
            *params
        )

        sse = np.sum(
            (pred - mos) ** 2
        )

        if sse < best_sse:
            best_sse = sse
            best_params = params

    except Exception as e:
        print("Fit failed:", b2, e)


assert best_params is not None

mapped = logistic5(
    quality_score,
    *best_params
)

plcc = pearsonr(
    mapped,
    mos
).statistic

rmse = np.sqrt(
    np.mean(
        (mapped - mos) ** 2
    )
)


paper = {
    "SROCC": 0.7579,
    "KROCC": 0.5623,
    "PLCC":  0.8226,
    "RMSE":  10.7534,
}


print("\n==============================")
print("MDLIVE CORRECTED METRICS")
print("==============================")

print(f"SROCC: {srocc:.6f}")
print(f"KROCC: {krocc:.6f}")
print(f"PLCC:  {plcc:.6f}")
print(f"RMSE:  {rmse:.6f}")

print("\nPaper:")
print("SROCC: 0.757900")
print("KROCC: 0.562300")
print("PLCC:  0.822600")
print("RMSE:  10.753400")

print("\nDifferences:")
print(f"SROCC: {srocc - paper['SROCC']:+.6f}")
print(f"KROCC: {krocc - paper['KROCC']:+.6f}")
print(f"PLCC:  {plcc  - paper['PLCC']:+.6f}")
print(f"RMSE:  {rmse  - paper['RMSE']:+.6f}")

In [ ]:
from pathlib import Path

report = """
MDFS MDLIVE REPRODUCTION
========================

Dataset:
MDLIVE / LIVE Multiply Distorted

Images evaluated:
450

Reference images:
15

Hardware:
2 x Tesla T4

Score orientation:
Raw MDFS higher = worse quality
MDLIVE subjective score higher = worse quality
Therefore raw correlations are directly paper-comparable.

Paper vs Reproduction
---------------------

SROCC
Paper: 0.757900
Ours:  0.757856
Diff: -0.000044

KROCC
Paper: 0.562300
Ours:  0.562316
Diff: +0.000016

PLCC
Paper: 0.822600
Ours:  0.822873
Diff: +0.000273

RMSE
Paper: 10.753400
Ours:  10.746272
Diff: -0.007128

Validation:
Rows: 450
Unique images: 450
All scores finite: True
Dual-GPU evaluation: True

Dual-GPU split:
GPU 0: 225
GPU 1: 225
"""

Path("reports/mdlive_reproduction.txt").write_text(report)

print(report)

In [ ]:
!cd /kaggle/working/MDFS && \
zip -r -q presentation_artifacts_6datasets.zip \
    reports \
    results \
    logs \
    scripts \
    src \
    tests

!ls -lh /kaggle/working/MDFS/presentation_artifacts_6datasets.zip

In [ ]:
!echo "===== REPORTS ====="
!find /kaggle/working/MDFS/reports -maxdepth 1 -type f -printf "%f\n" | sort

!echo
!echo "===== IMPORTANT RESULTS ====="
!find /kaggle/working/MDFS/results -maxdepth 1 -type f -printf "%f\n" | sort

In [1]:
%cd /kaggle/working/MDFS

/kaggle/working/MDFS


In [2]:
!mkdir -p reports

!sha256sum artifacts/MDFS_weights_rebuilt.pth \
> reports/MDFS_weights_rebuilt.sha256

!cat reports/MDFS_weights_rebuilt.sha256

c3fa643b21fccdf4152895c24b69e98efcf226cfa76ae9cce7176614e7b464de  artifacts/MDFS_weights_rebuilt.pth


In [4]:
!zip -r -q mdfs_6datasets_reproduction_final_bundle.zip \
    README.md \
    requirements.txt \
    model.py \
    artifacts \
    figures \
    test_rebuilt_smoke.py \
    train_benchmark10.py \
    train.py \
    test.py \
    src \
    scripts \
    tests \
    reports \
    results \
    logs \
    docs \
    -x "*/__pycache__/*" "*.pyc"

!ls -lh mdfs_6datasets_reproduction_bundle.zip

-rw-r--r-- 1 root root 1.4M Sep 20 12:41 mdfs_6datasets_reproduction_bundle.zip
